In [77]:
import sys
import os
# only go up a folder if we are currently inside the notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    
project_root = os.getcwd() 
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [78]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Phase 1: Verification
Verification if all files are imported successfully.

In [79]:
import pandas as pd

In [80]:
df = pd.read_csv("csv/tokopedia_product_reviews_2025.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65543 entries, 0 to 65542
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   review_text       65543 non-null  str  
 1   review_date       65543 non-null  str  
 2   review_id         65543 non-null  int64
 3   product_name      65543 non-null  str  
 4   product_category  65543 non-null  str  
 5   product_variant   26749 non-null  str  
 6   product_price     65543 non-null  int64
 7   product_url       65543 non-null  str  
 8   product_id        65543 non-null  int64
 9   rating            65543 non-null  int64
 10  sold_count        65543 non-null  int64
 11  shop_id           65543 non-null  int64
 12  sentiment_label   65543 non-null  str  
dtypes: int64(6), str(7)
memory usage: 6.5 MB


65543 rows and 13 columns, column names seems correct as well. product_variant has missing values, but doesn't matter for now.

In [81]:
df.head(5)

,review_text,review_date,review_id,product_name,product_category,product_variant,product_price,product_url,product_id,rating,sold_count,shop_id,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,2024-12-22,1134256160,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,2025-02-25,1242584634,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,2025-07-15,1573444677,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
3,Telor sudah diterima dengan baik dan tidak ada...,2025-07-20,1581728541,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
4,"Alhamdulillah penjual amanah,Telor nya terbaik...",2023-04-24,881041355,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Full Design,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive


In [82]:
time_series_df = df[["review_date", "sentiment_label"]]
time_series_df

,review_date,sentiment_label
0,2024-12-22,positive
1,2025-02-25,positive
2,2025-07-15,positive
3,2025-07-20,positive
4,2023-04-24,positive
...,...,...
65538,2025-01-06,positive
65539,2025-05-08,positive
65540,2025-05-08,positive
65541,2025-02-19,positive


In [88]:
valid_sentiments = ["positive"]

In [89]:
~time_series_df["sentiment_label"].isin(valid_sentiments)

0        False
1        False
2        False
3        False
4        False
         ...  
65538    False
65539    False
65540    False
65541    False
65542    False
Name: sentiment_label, Length: 65543, dtype: bool

In [92]:
time_series_df.loc[~time_series_df["sentiment_label"].isin(valid_sentiments), "sentiment_label"]

12       negative
37        neutral
46        neutral
49       negative
51       negative
           ...   
64945    negative
65181     neutral
65349    negative
65390     neutral
65425     neutral
Name: sentiment_label, Length: 1600, dtype: str

In [93]:
valid_sentiments = ["neutral", "negative"]
time_series_df.loc[~time_series_df["sentiment_label"].isin(valid_sentiments), "sentiment_label"] = "test"
time_series_df

,review_date,sentiment_label
0,2024-12-22,test
1,2025-02-25,test
2,2025-07-15,test
3,2025-07-20,test
4,2023-04-24,test
...,...,...
65538,2025-01-06,test
65539,2025-05-08,test
65540,2025-05-08,test
65541,2025-02-19,test


In [38]:
pd.to_datetime(time_series_df["review_date"])

0       2024-12-22
1       2025-02-25
2       2025-07-15
3       2025-07-20
4       2023-04-24
           ...    
65538   2025-01-06
65539   2025-05-08
65540   2025-05-08
65541   2025-02-19
65542   2024-11-21
Name: review_date, Length: 65543, dtype: datetime64[us]

In [39]:
time_series_df["review_date"] = pd.to_datetime(time_series_df["review_date"])
time_series_df

,review_date,sentiment_label
0,2024-12-22,positive
1,2025-02-25,positive
2,2025-07-15,positive
3,2025-07-20,positive
4,2023-04-24,positive
...,...,...
65538,2025-01-06,positive
65539,2025-05-08,positive
65540,2025-05-08,positive
65541,2025-02-19,positive


In [40]:
time_series_df.groupby(["review_date", "sentiment_label"]).size().reset_index(name="review_count").sort_values(["review_date", "sentiment_label"])

,review_date,sentiment_label,review_count
0,2015-11-18,positive,1
1,2015-12-23,positive,1
2,2015-12-24,positive,1
3,2016-02-07,positive,1
4,2016-02-19,positive,1
...,...,...,...
4062,2025-12-10,neutral,2
4063,2025-12-10,positive,65
4064,2025-12-11,neutral,2
4065,2025-12-11,positive,60


In [41]:
ts_df_counts = time_series_df.groupby(["review_date", "sentiment_label"]).size().reset_index(name="review_count").sort_values(["review_date", "sentiment_label"])
ts_df_counts

,review_date,sentiment_label,review_count
0,2015-11-18,positive,1
1,2015-12-23,positive,1
2,2015-12-24,positive,1
3,2016-02-07,positive,1
4,2016-02-19,positive,1
...,...,...,...
4062,2025-12-10,neutral,2
4063,2025-12-10,positive,65
4064,2025-12-11,neutral,2
4065,2025-12-11,positive,60


In [74]:
ts_df_counts.groupby("sentiment_label")["review_count"].cumsum()

0           1
1           2
2           3
3           4
4           5
        ...  
4062      800
4063    63882
4064      802
4065    63942
4066    63943
Name: review_count, Length: 4067, dtype: int64

In [73]:
ts_df_counts["cumulative_count"] = ts_df_counts.groupby("sentiment_label")["review_count"].cumsum()
ts_df_counts

,review_date,sentiment_label,review_count,cumulative_count
0,2015-11-18,positive,1,1
1,2015-12-23,positive,1,2
2,2015-12-24,positive,1,3
3,2016-02-07,positive,1,4
4,2016-02-19,positive,1,5
...,...,...,...,...
4062,2025-12-10,neutral,2,800
4063,2025-12-10,positive,65,63882
4064,2025-12-11,neutral,2,802
4065,2025-12-11,positive,60,63942
